# Gold Layer — Attendance Star Schema

Builds five dimensions and one fact in **`coptic_analytics.gold`** from **`coptic_analytics.silver`**. Run the cells in order — the fact joins two of the dimensions.

- **`dim_date`** — one row per calendar day, 2023-09-01 to 2026-12-31, generated rather than extracted so the range is contiguous; carries the academic year (Sep–Aug).
- **`dim_church`** — one row per church, with diocese flattened in and the priest's own absence threshold in weeks.
- **`dim_class`** — one row per class, with its department flattened in; names repeat across churches, so the id is the identity and the name is the grouper.
- **`dim_member`** — one row per member, with family and family head flattened in for the follow-up list.
- **`dim_activity`** — one row per activity (Liturgy, Sunday School, Trip…), with its engagement weight and whether attendance is mandatory.
- **`fact_attendance`** — one row per rostered member per event: the **expected**-attendance grid, not a copy of `silver.attendance`, so a session nobody recorded shows up as `not_marked` instead of vanishing.

### Dim Date 

In [0]:
%sql

create or replace table coptic_analytics.gold.dim_date
using delta
as

with calendar as (
    select explode(sequence(date'2023-09-01', date'2026-12-31', interval 1 day)) as full_date
),

days as (
    select
        cast(date_format(full_date, 'yyyyMMdd') as int)                     as date_key,
        full_date,
        year(full_date)                                                     as year,
        quarter(full_date)                                                  as quarter,
        month(full_date)                                                    as month,
        date_format(full_date, 'MMMM')                                      as month_name,
        date_format(full_date, 'MMM yyyy')                                  as month_year,
        weekofyear(full_date)                                               as iso_week,
        dayofmonth(full_date)                                               as day_of_month,
        date_format(full_date, 'EEEE')                                      as day_name,
        extract(DAYOFWEEK_ISO from full_date)                               as day_of_week
    from calendar
)

select
    date_key,
    full_date,
    year,
    quarter,
    month,
    month_name,
    month_year,
    iso_week,
    day_of_month,
    day_name,
    day_of_week
from days;

### Dim Church

In [0]:
%sql

create or replace table coptic_analytics.gold.dim_church
using delta 
as 

select
    c.church_id,
    c.church_name,
    c.city,
    c.country,
    c.absence_threshold_weeks,
    c.timezone,
    c.diocese_id,
    d.diocese_name,
    d.country    as diocese_country
from coptic_analytics.silver.church c
join coptic_analytics.silver.diocese d
    on d.diocese_id = c.diocese_id;

### Dim Class

In [0]:
%sql

create or replace table coptic_analytics.gold.dim_class
using delta
as

select
    cl.class_id,
    cl.class_name,
    cl.department_id,
    d.department_name,
    cl.church_id,
    ch.church_name
from coptic_analytics.silver.class cl
join coptic_analytics.silver.department d
    on d.department_id = cl.department_id
   and d.church_id     = cl.church_id
join coptic_analytics.silver.church ch
    on ch.church_id = cl.church_id;

### Dim Member

In [0]:
%sql

create or replace table coptic_analytics.gold.dim_member
using delta 
as

with live_member as (
    select * from coptic_analytics.silver.member
),

family_head as (
    select family_id, member_name as family_head_name
    from live_member
    where is_head_of_family
)

select
    m.member_id,
    m.member_name,
    m.gender,
    m.date_of_birth,
    cast(floor(months_between(current_date(), m.date_of_birth) / 12) as int)
                                                 as age_years,
    month(m.date_of_birth)                       as birth_month,
    dayofmonth(m.date_of_birth)                  as birth_day,
    m.joined_date,
    m.left_date,
    m.left_date is null                          as is_current_member,
    m.family_id,
    coalesce(f.family_name, 'Unknown family')    as family_name,
    m.is_head_of_family,
    h.family_head_name,
    m.church_id
from live_member m
left join coptic_analytics.silver.family f
    on f.family_id = m.family_id
left join family_head h
    on h.family_id = m.family_id;

### Dim Activity

In [0]:
%sql

create or replace table coptic_analytics.gold.dim_activity
select
    a.activity_id,
    a.activity_name,
    a.is_mandatory,
    a.weight,
    a.church_id
from coptic_analytics.silver.activity a

### Fact Attendance

In [0]:
%sql

create or replace table coptic_analytics.gold.fact_attendance
using delta 
as

with roster as (
    select
        e.event_id,
        e.church_id,
        e.activity_id,
        e.class_id,
        e.event_date,
        mc.member_id
    from coptic_analytics.silver.event e
    join coptic_analytics.gold.dim_class dc
        on dc.class_id = e.class_id
    join coptic_analytics.silver.member_class mc
        on mc.class_id = e.class_id
       and mc.valid_from <= e.event_date
       and (mc.valid_to is null or mc.valid_to > e.event_date)
),

rostered_members as (
    select r.*
    from roster r
    join coptic_analytics.gold.dim_member dm
        on dm.member_id = r.member_id
),

outcome as (
    select
        rm.*,
        coalesce(a.status, 'not_marked') as attendance_status
    from rostered_members rm
    left join coptic_analytics.silver.attendance a
        on a.event_id  = rm.event_id
       and a.member_id = rm.member_id
)

select
    o.event_id,
    cast(date_format(o.event_date, 'yyyyMMdd') as int)  as date_key,
    o.event_date,
    o.church_id,
    o.class_id,
    o.activity_id,
    o.member_id,
    o.attendance_status
from outcome o